In [10]:
from pathlib import Path
import json
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import nltk
import re

In [12]:
# Load the ToxicBERT model
model_name = "unitary/toxic-bert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)


In [13]:
# Define project and data paths
PROJECT_ROOT = Path("/Users/tildeidunsloth/Desktop/Thesis")
DATA_DIR = PROJECT_ROOT / "data/raw"

fantasy_data_path = DATA_DIR / "generated_stories_fantasy.jsonl"
romance_data_path = DATA_DIR / "generated_stories_romance.jsonl"
litterary_fiction_data_path = DATA_DIR / "generated_stories_litterary_fiction.jsonl"

In [14]:
# Read JSONL file
with open(romance_data_path, 'r') as f:
    romance_stories = [json.loads(line) for line in f]

In [15]:
def split_sentences(text):
    # Split on periods, question marks, exclamation marks
    sentences = re.split(r'(?<=[.!?]) +', text)
    return sentences

In [16]:
# Function to get toxicity score for a sentence
def get_toxicity_score(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    # 0 = non-toxic, 1 = toxic
    scores = torch.softmax(outputs.logits, dim=1)
    toxic_score = scores[0][1].item()  # probability of toxic
    return toxic_score

In [21]:
# Process all stories and append average toxicity
for story_dict in romance_stories:
    story_text = story_dict["story"]
    sentences = split_sentences(story_text)
    sentence_scores = [get_toxicity_score(s) for s in sentences]
    avg_score = sum(sentence_scores) / len(sentence_scores) if sentence_scores else 0
    # Append the average toxicity to the story dictionary
    story_dict["average_toxicity"] = avg_score

# Check results
for s in romance_stories:
    print(s)

{'story': "Beginning\n\nElena had cultivated precision the way other people cultivated gardens: with careful pruning, exact measurements, and a small, private ruthlessness. For ten years she had lived inside a stainless-steel kingdom—an open kitchen in a city that never truly slept, where the heat of the line was a constant and the applause for a tasting menu came as quickly as the next reservation. She could file a perfect brunoise of carrot with one glance, could coax a sauce into velvet silence with the same hand that had, once, typed the love letters she had sent and received in her twenties.\n\nWhen she finally booked a week on the island of Isola Verde, it was not bravado so much as exhaustion. The restaurant had been granted a new Michelin star and with it a roster of interviews and invitations and obligations Elena no longer possessed energy for. She wanted clay under her nails—something visceral, unmeasured. She wanted seeds to be seeds again instead of numbers on a procuremen